In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as F

# CONFIGURAÇÕES GLOBAIS
db_name = "demo"
parquet_table_name = "titles_parquet"
hdfs_path = "hdfs://hdfs-nn:9000/demo/bronze/titles.csv"
output_path = "hdfs://hdfs-nn:9000/demo/silver/titles_parquet"
warehouse_location = 'hdfs://hdfs-nn:9000/demo/silver/parquet_warehouse' 

# 1. CONFIGURAR SPARK (Com suporte Hive)
spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("Titles Bronze to Parquet Silver") \
    .config("spark.sql.warehouse.dir", warehouse_location) \
    .config("hive.metastore.uris", "thrift://hive-metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

print("SparkSession iniciada com suporte Hive.")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {db_name}")


# 2. LER CSV E LIMPEZA DE QUALIDADE

# A. Ler CSV
titles_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("escape", '"') \
    .csv(hdfs_path)

# B. Aplica correção de tipos
titles_df = titles_df \
    .withColumn("seasons", F.col("seasons").cast("int")) \
    .withColumn("imdb_score", F.col("imdb_score").cast("double"))

# C. LIMPEZA DE NULOS E STRINGS VAZIAS (Substitui null e '' por 'Unknown')
substituicao_str = "Unknown"
string_cols = ["id", "title", "type", "description", "age_certification", 
               "genres", "production_countries", "imdb_id"]

# Substituir valores nulos (Nulls)
titles_df = titles_df.fillna(substituicao_str, subset=string_cols)

# Substituir strings vazias ('')
for col_name in string_cols:
    titles_df = titles_df.withColumn(
        col_name,
        F.when(F.col(col_name) == "", substituicao_str).otherwise(F.col(col_name))
    )
print("Limpeza de nulos/vazios concluída nos campos de texto.")


# 3. TRANSFORMAÇÕES (Limpeza de Géneros e One-Hot Encoding)

# 3.1. Limpeza de Coluna 'genres'
titles_df = titles_df.withColumn(
    "genres",
    F.regexp_replace(F.col("genres"), "^\\[|]|\'", "") 
)
print("Géneros limpos.")

# 3.2. Detetar e Recolher todos os Países Únicos (Lógica Dinâmica)
print("A detetar países únicos para One-Hot Encoding...")
country_processing_df = titles_df.select(
    F.regexp_replace(F.col("production_countries"), r"\[|]|\'", "").alias("country_list")
).filter(F.col("country_list") != "Unknown") # Filtra o 'Unknown' para não criar is_Unknown_production

all_countries = (country_processing_df
    .withColumn("country_code_exploded", F.explode(F.split(F.col("country_list"), ", ")))
    .select(F.trim(F.col("country_code_exploded")).alias("country_code"))
    .filter(F.col("country_code").isNotNull() & (F.length(F.col("country_code")) == 2))
    .distinct()
    .rdd.map(lambda row: row[0]).collect() 
)
print(f"Número total de países únicos detetados: {len(all_countries)}")

# 3.3. Aplicar One-Hot Encoding Dinâmico e Remover a Coluna Original
for country_code in all_countries:
    titles_df = titles_df.withColumn(
        f"is_{country_code}_production",
        F.when(F.col("production_countries").like(f"%'{country_code}'%"), 1).otherwise(0).cast("int")
    )
titles_df = titles_df.drop("production_countries") 
print("One-Hot Encoding aplicado.")

# ---------------------------------------------------------
# 4. ESCREVER PARQUET PARTICIONADO E CRIAR TABELA HIVE
# ---------------------------------------------------------

# Sobrescrever a pasta de dados Parquet
titles_df.write \
    .mode("overwrite") \
    .partitionBy("release_year") \
    .format("parquet") \
    .save(output_path)

print(f"\nParquet escrito com sucesso em {output_path} e particionado por release_year.")

# Registar a tabela no Hive (Corrigido o erro de AnalysisException)
spark.sql(f"DROP TABLE IF EXISTS {db_name}.{parquet_table_name}")

# Usamos a sintaxe USING PARQUET para inferir o esquema de todas as 107 colunas diretamente dos ficheiros.
spark.sql(f"""
    CREATE EXTERNAL TABLE {db_name}.{parquet_table_name}
    USING PARQUET
    LOCATION '{output_path}'
""")

# A recuperação de partições é essencial para que o Hive descubra as pastas 'release_year=XXXX'
spark.catalog.recoverPartitions(f"{db_name}.{parquet_table_name}")

print(f"Tabela Hive '{parquet_table_name}' criada com o esquema completo e partições recuperadas!")

# 5. CONSULTA DE TESTE
print("\nConsulta de teste da tabela Parquet (A verificar a limpeza de nulos):")
spark.sql("""
    SELECT 
        title, 
        genres, 
        age_certification, 
        imdb_score, 
        release_year
    FROM demo.titles_parquet
    WHERE age_certification = 'Unknown' OR title = 'Unknown'
    LIMIT 5
""").show(truncate=False)

spark.stop()

SparkSession iniciada com suporte Hive.
Limpeza de nulos/vazios concluída nos campos de texto.
Géneros limpos.
A detetar países únicos para One-Hot Encoding...
Número total de países únicos detetados: 108
One-Hot Encoding aplicado.

Parquet escrito com sucesso em hdfs://hdfs-nn:9000/demo/silver/titles_parquet e particionado por release_year.
Tabela Hive 'titles_parquet' criada com o esquema completo e partições recuperadas!

Consulta de teste da tabela Parquet (A verificar a limpeza de nulos):
+------------------------------+---------------------------------+-----------------+----------+------------+
|title                         |genres                           |age_certification|imdb_score|release_year|
+------------------------------+---------------------------------+-----------------+----------+------------+
|The Platform                  |scifi, thriller, horror, european|Unknown          |7.0       |2019        |
|Lunana: A Yak in the Classroom|drama, family                    

In [ ]:
spark.stop()